# Models & theory: practice notebook

15 exercises, Easy to Hard, each with hidden tests.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("m1")` prints the reference solution for that exercise.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, pandas, scipy, scikit-learn and matplotlib)
import importlib.util, subprocess, sys
for _p in []:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The datasets every exercise uses. Read them if you like; don't change them.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
_g = np.random.default_rng(2026)
X = _g.normal(size=(300, 3))
y = 1.5 + X @ np.array([2.0, -1.0, 0.5]) + _g.normal(0, 1, 300)
XL = _g.normal(size=(200, 10))
XL[:, 1] = 0.8 * XL[:, 0] + 0.6 * XL[:, 1]
yL = 2.0 + XL @ np.array([3, -2, 0, 0, 1.5, 0, 0, 0, 0, 0.0]) + _g.normal(0, 1, 200)
XC = _g.normal(size=(600, 3))
yC = (_g.uniform(size=600) < 1 / (1 + np.exp(-(-0.5 + XC @ np.array([1.2, -0.8, 0.3]))))).astype(int)
_f = _g.normal(size=(400, 2))
Z = _f @ _g.normal(size=(2, 6)) + 0.3 * _g.normal(size=(400, 6))
_T = ["AAA", "BBB", "CCC", "DDD"]
_cov = 0.0001 * np.array([[4, 2, 1, 0.5], [2, 3, 1, 0.5], [1, 1, 2, 0.2], [0.5, 0.5, 0.2, 1]])
_r = _g.multivariate_normal([0.0006, 0.0004, 0.0003, 0.0001], _cov, 500)
_r[_g.uniform(size=_r.shape) < 0.01] = np.nan
rets = pd.DataFrame(_r, index=pd.bdate_range("2024-01-01", periods=500), columns=_T)
_m = 2000
credit = pd.DataFrame({
    "income": np.round(_g.lognormal(11, 0.4, _m), -2),
    "debt_ratio": np.round(_g.beta(2, 5, _m), 3),
    "age": _g.integers(21, 70, _m),
    "late_payments": _g.poisson(0.6, _m),
})
_z = -1.0 - 0.00002 * (credit.income - 60000) + 3.0 * credit.debt_ratio - 0.02 * (credit.age - 40) + 0.7 * credit.late_payments
credit["default"] = (_g.uniform(size=_m) < 1 / (1 + np.exp(-_z))).astype(int)
mu = np.array([0.06, 0.08, 0.10, 0.04, 0.12])
_s = np.array([0.15, 0.20, 0.25, 0.08, 0.30])
_rho = np.array([[1, .5, .3, .1, .4], [.5, 1, .4, .1, .5], [.3, .4, 1, 0, .6], [.1, .1, 0, 1, 0], [.4, .5, .6, 0, 1]])
Sigma = np.outer(_s, _s) * _rho
_lv = np.cumsum(_g.normal(0, 0.3, 200)) + 10
level_obs = _lv + _g.normal(0, 1.0, 200)
level_obs[[40, 41, 42, 120]] = np.nan
hx = 50 + np.cumsum(_g.normal(0, 1, 300))
_beta = 1.2 + np.cumsum(_g.normal(0, 0.01, 300))
hy = _beta * hx + _g.normal(0, 1.5, 300)
del _g

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("...").decode())
_SETUP = _b64.b64decode("CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCl9nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDIwMjYpClggPSBfZy5ub3JtYWwoc2l6ZT0oMzAwLCAzKSkKeSA9IDEuNSArIFggQCBucC5hcnJheShbMi4wLCAtMS4wLCAwLjVdKSArIF9nLm5vcm1hbCgwLCAxLCAzMDApClhMID0gX2cubm9ybWFsKHNpemU9KDIwMCwgMTApKQpYTFs6LCAxXSA9IDAuOCAqIFhMWzosIDBdICsgMC42ICogWExbOiwgMV0KeUwgPSAyLjAgKyBYTCBAIG5wLmFycmF5KFszLCAtMiwgMCwgMCwgMS41LCAwLCAwLCAwLCAwLCAwLjBdKSArIF9nLm5vcm1hbCgwLCAxLCAyMDApClhDID0gX2cubm9ybWFsKHNpemU9KDYwMCwgMykpCnlDID0gKF9nLnVuaWZvcm0oc2l6ZT02MDApIDwgMSAvICgxICsgbnAuZXhwKC0oLTAuNSArIFhDIEAgbnAuYXJyYXkoWzEuMiwgLTAuOCwgMC4zXSkpKSkpLmFzdHlwZShpbnQpCl9mID0gX2cubm9ybWFsKHNpemU9KDQwMCwgMikpClogPSBfZiBAIF9nLm5vcm1hbChzaXplPSgyLCA2KSkgKyAwLjMgKiBfZy5ub3JtYWwoc2l6ZT0oNDAwLCA2KSkKX1QgPSBbIkFBQSIsICJCQkIiLCAiQ0NDIiwgIkRERCJdCl9jb3YgPSAwLjAwMDEgKiBucC5hcnJheShbWzQsIDIsIDEsIDAuNV0sIFsyLCAzLCAxLCAwLjVdLCBbMSwgMSwgMiwgMC4yXSwgWzAuNSwgMC41LCAwLjIsIDFdXSkKX3IgPSBfZy5tdWx0aXZhcmlhdGVfbm9ybWFsKFswLjAwMDYsIDAuMDAwNCwgMC4wMDAzLCAwLjAwMDFdLCBfY292LCA1MDApCl9yW19nLnVuaWZvcm0oc2l6ZT1fci5zaGFwZSkgPCAwLjAxXSA9IG5wLm5hbgpyZXRzID0gcGQuRGF0YUZyYW1lKF9yLCBpbmRleD1wZC5iZGF0ZV9yYW5nZSgiMjAyNC0wMS0wMSIsIHBlcmlvZHM9NTAwKSwgY29sdW1ucz1fVCkKX20gPSAyMDAwCmNyZWRpdCA9IHBkLkRhdGFGcmFtZSh7CiAgICAiaW5jb21lIjogbnAucm91bmQoX2cubG9nbm9ybWFsKDExLCAwLjQsIF9tKSwgLTIpLAogICAgImRlYnRfcmF0aW8iOiBucC5yb3VuZChfZy5iZXRhKDIsIDUsIF9tKSwgMyksCiAgICAiYWdlIjogX2cuaW50ZWdlcnMoMjEsIDcwLCBfbSksCiAgICAibGF0ZV9wYXltZW50cyI6IF9nLnBvaXNzb24oMC42LCBfbSksCn0pCl96ID0gLTEuMCAtIDAuMDAwMDIgKiAoY3JlZGl0LmluY29tZSAtIDYwMDAwKSArIDMuMCAqIGNyZWRpdC5kZWJ0X3JhdGlvIC0gMC4wMiAqIChjcmVkaXQuYWdlIC0gNDApICsgMC43ICogY3JlZGl0LmxhdGVfcGF5bWVudHMKY3JlZGl0WyJkZWZhdWx0Il0gPSAoX2cudW5pZm9ybShzaXplPV9tKSA8IDEgLyAoMSArIG5wLmV4cCgtX3opKSkuYXN0eXBlKGludCkKbXUgPSBucC5hcnJheShbMC4wNiwgMC4wOCwgMC4xMCwgMC4wNCwgMC4xMl0pCl9zID0gbnAuYXJyYXkoWzAuMTUsIDAuMjAsIDAuMjUsIDAuMDgsIDAuMzBdKQpfcmhvID0gbnAuYXJyYXkoW1sxLCAuNSwgLjMsIC4xLCAuNF0sIFsuNSwgMSwgLjQsIC4xLCAuNV0sIFsuMywgLjQsIDEsIDAsIC42XSwgWy4xLCAuMSwgMCwgMSwgMF0sIFsuNCwgLjUsIC42LCAwLCAxXV0pClNpZ21hID0gbnAub3V0ZXIoX3MsIF9zKSAqIF9yaG8KX2x2ID0gbnAuY3Vtc3VtKF9nLm5vcm1hbCgwLCAwLjMsIDIwMCkpICsgMTAKbGV2ZWxfb2JzID0gX2x2ICsgX2cubm9ybWFsKDAsIDEuMCwgMjAwKQpsZXZlbF9vYnNbWzQwLCA0MSwgNDIsIDEyMF1dID0gbnAubmFuCmh4ID0gNTAgKyBucC5jdW1zdW0oX2cubm9ybWFsKDAsIDEsIDMwMCkpCl9iZXRhID0gMS4yICsgbnAuY3Vtc3VtKF9nLm5vcm1hbCgwLCAwLjAxLCAzMDApKQpoeSA9IF9iZXRhICogaHggKyBfZy5ub3JtYWwoMCwgMS41LCAzMDApCmRlbCBfZwo=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. OLS from the normal equations
*Easy · id `m1`*

<p>Write <code>ols(X, y)</code> for an <code>(n, p)</code> array <code>X</code> <b>without</b> an intercept column. Add the intercept yourself and return β̂ of length p + 1, intercept first, by solving the normal equations.</p>
<p>If the design matrix with intercept doesn't have full column rank, raise <code>ValueError</code> instead of returning numbers.</p>
<p class=hint>Use <code>np.linalg.solve</code>, not <code>inv</code>. Check the rank with <code>np.linalg.matrix_rank</code>.</p>

**Theory to know cold**
<ul><li>Derive β̂ = (XᵀX)⁻¹Xᵀy by setting the gradient of ‖y − Xβ‖² to zero, and show it is a minimum (XᵀX is positive semi-definite).</li>
<li><b>Necessary</b> for a unique β̂: full column rank (no perfect collinearity, n ≥ p + 1). That's all the algebra needs.</li>
<li><b>Sufficient</b> for unbiasedness: linearity + E[ε|X] = 0. Add homoskedastic, uncorrelated errors for Gauss-Markov (BLUE). Add normal errors for exact t and F tests in small samples; large n replaces normality via the CLT.</li>
<li>Geometry: ŷ is the projection of y onto the column space of X; residuals are orthogonal to every column.</li></ul>

In [ ]:
def ols(X, y):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    print(ols(X, y))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m1")

---
## 2. Classical vs robust standard errors
*Medium · id `m2`*

<p>Write <code>ols_se(X, y)</code> (X without intercept) returning <code>(beta, se_classical, se_hc1)</code>, all of length p + 1, intercept first.</p>
<ul><li>Classical: s²(XᵀX)⁻¹ with s² = eᵀe / (n − k).</li>
<li>HC1 (White, small-sample corrected): (XᵀX)⁻¹ Xᵀ diag(e²) X (XᵀX)⁻¹ · n / (n − k).</li></ul>
<p class=hint>k = p + 1. Form Xᵀ diag(e²) X as <code>(A * e[:, None]**2).T @ A</code> without building an n × n matrix.</p>

**Theory to know cold**
<ul><li>Heteroskedasticity leaves β̂ unbiased and consistent, so <b>prediction is fine</b>, but the classical SE is wrong, so <b>inference breaks</b>. Robust (sandwich) SEs fix inference without changing β̂.</li>
<li>Autocorrelated errors (time series): same story, use Newey-West (HAC). Clustered data: cluster-robust SEs.</li>
<li>Know why the sandwich has that shape: Var(β̂) = (XᵀX)⁻¹ Xᵀ Var(ε) X (XᵀX)⁻¹.</li></ul>

In [ ]:
def ols_se(X, y):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    b, sc, sr = ols_se(X, y)
    print(b); print(sc); print(sr)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m2")

---
## 3. Which violation breaks inference?
*Medium · id `m3`*

<p>Show by simulation that heteroskedasticity breaks classical confidence intervals but not robust ones.</p>
<p>Write <code>het_coverage(n=200, reps=2000, seed=0)</code>. In each rep draw <code>x = rng.uniform(0, 3, n)</code>, then <code>e = rng.normal(size=n) * x**1.5</code>, and set y = 1 + 2x + e. Fit OLS, build the 95% CI for the slope with the t critical value (n − 2 df), once with the classical SE and once with HC1. Return <code>(coverage_classical, coverage_hc1)</code>: the share of reps whose CI contains 2.</p>
<p class=hint>A loop over reps is fine here. You can reuse your formulas from problem 2.</p>

**Theory to know cold**
<ul><li><b>Breaks inference, not prediction:</b> heteroskedasticity, autocorrelation, non-normal errors in small samples, multicollinearity (huge SEs).</li>
<li><b>Breaks both</b> (biased β̂): omitted variables correlated with X, measurement error in X (attenuation), simultaneity or reverse causality, wrong functional form.</li>
<li><b>Breaks prediction but not the fitted causal story:</b> distribution shift between train and test, non-stationarity, leakage.</li>
<li>Inference asks "is β̂ right and how sure am I?" Prediction asks "is ŷ good out of sample?" A biased model can predict well, and an unbiased one can predict badly.</li></ul>

In [ ]:
def het_coverage(n=200, reps=2000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(het_coverage(reps=1000))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m3")

---
## 4. Collinearity: bad coefficients, fine predictions
*Medium · id `m4`*

<p>Write <code>collinearity(rho, n=100, reps=1000, seed=0)</code>. In each rep draw <code>z1, z2 = rng.normal(size=(2, n))</code>, set x1 = z1, x2 = ρ·z1 + √(1−ρ²)·z2 and y = 1 + x1 + x2 + N(0, 1) noise (draw it after z1, z2). Fit OLS of y on x1, x2.</p>
<p>Return <code>(sd_beta1, sd_pred)</code>: the standard deviation across reps of β̂₁, and of the prediction at x1 = x2 = 1. Then compare ρ = 0 with ρ = 0.99.</p>

**Theory to know cold**
<ul><li>Collinearity is not a violation of Gauss-Markov: β̂ stays unbiased, but Var(β̂ⱼ) = σ² / (n·Var(xⱼ)·(1 − R²ⱼ)) blows up. VIF = 1 / (1 − R²ⱼ).</li>
<li>Predictions along the direction the data covers stay precise. Predictions off that direction (x1 high, x2 low) don't.</li>
<li>Fixes for inference: more data, drop or combine variables, ridge (trades bias for variance), PCA regression.</li></ul>

In [ ]:
def collinearity(rho, n=100, reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    for r in (0, 0.9, 0.99):
        print(r, collinearity(r, reps=500))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m4")

---
## 5. Ridge in closed form
*Medium · id `m5`*

<p>Write <code>ridge(X, y, lam)</code> minimising ‖y − b₀ − Xb‖² + λ‖b‖², with the intercept b₀ <b>not</b> penalised. Return <code>(intercept, coef)</code>. The tests check you match scikit-learn's <code>Ridge(alpha=lam)</code>.</p>
<p class=hint>Centre X and y, solve (XcᵀXc + λI) b = Xcᵀyc, then b₀ = ȳ − x̄ᵀb.</p>

**Theory to know cold**
<ul><li>Derive it: gradient −2Xᵀ(y − Xb) + 2λb = 0. XᵀX + λI is invertible for any λ &gt; 0, so ridge works even when p &gt; n or columns are collinear.</li>
<li>Shrinkage: in the SVD basis each component is scaled by d²/(d² + λ). Biased, lower variance; the bias-variance trade-off picks λ (cross-validation).</li>
<li>Standardise features first, or the penalty hits features unevenly. Don't penalise the intercept.</li>
<li>Bayesian view: ridge is the posterior mean under a N(0, τ²I) prior with λ = σ²/τ² (problem 13).</li></ul>

In [ ]:
def ridge(X, y, lam):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    for lam in (0, 10, 100, 1000):
        print(lam, ridge(XL, yL, lam)[1].round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m5")

---
## 6. Lasso by coordinate descent
*Hard · id `m6`*

<p>Lasso has no closed form. Write <code>lasso(X, y, lam, n_iter=500)</code> minimising (1/2n)‖y − b₀ − Xb‖² + λ‖b‖₁ by cyclic coordinate descent, intercept unpenalised. Return <code>(intercept, coef)</code>. The tests compare with scikit-learn's <code>Lasso(alpha=lam)</code> and check which coefficients are exactly zero.</p>
<p class=hint>Centre first. For coordinate j, with partial residual r₍₋ⱼ₎: ρⱼ = xⱼᵀr₍₋ⱼ₎, then bⱼ = S(ρⱼ, nλ) / ‖xⱼ‖², where S(z, t) = sign(z)·max(|z| − t, 0).</p>

**Theory to know cold**
<ul><li>Derive the soft-threshold update from the subgradient (KKT) condition: 0 ∈ −xⱼᵀ(y − Xb)/n + λ·∂|bⱼ|.</li>
<li>Why ℓ1 gives exact zeros and ℓ2 doesn't: the diamond-shaped constraint has corners on the axes.</li>
<li>With orthonormal X, lasso = soft-thresholded OLS, ridge = scaled OLS, best subset = hard-thresholded OLS.</li>
<li>With correlated features lasso picks one somewhat arbitrarily; elastic net mixes ℓ1 and ℓ2 to keep groups. Post-selection inference on lasso coefficients is not valid by default.</li></ul>

In [ ]:
def lasso(X, y, lam, n_iter=500):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    for lam in (0.01, 0.1, 0.5):
        print(lam, lasso(XL, yL, lam)[1].round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m6")

---
## 7. Gradient descent for logistic regression
*Medium · id `m7`*

<p>Write <code>logistic_gd(X, y, lr=0.5, n_iter=3000)</code>: fit an unregularised logistic regression with intercept by batch gradient descent on the <b>mean</b> log-loss, starting from zeros. Return <code>(intercept, coef)</code>.</p>
<p>Then press Run: it tries several learning rates. Explain what happens at each.</p>
<p class=hint>Gradient of the mean log-loss: Xᵀ(σ(Xw) − y) / n, with an intercept column in X.</p>

**Theory to know cold**
<ul><li>Derive the gradient of the log-loss; show the loss is convex (Hessian XᵀSX/n, S = diag(p(1 − p)) ⪰ 0).</li>
<li>Step size: for an L-smooth convex loss, η &lt; 2/L converges; too large oscillates or diverges. Logistic loss has L ≤ λmax(XᵀX)/(4n).</li>
<li>Rates: GD on smooth convex is O(1/k), strongly convex is linear. Newton converges quadratically near the optimum but costs O(p³) per step.</li>
<li>Variants: SGD and mini-batch (noisy, cheap, needs a decaying step), momentum, Adam. Feature scaling changes the conditioning and so the speed.</li>
<li>Perfectly separable data: the MLE doesn't exist, weights grow forever; regularise.</li></ul>

In [ ]:
def logistic_gd(X, y, lr=0.5, n_iter=3000):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    def loss(b0, b):
        p = 1 / (1 + np.exp(-(b0 + XC @ b)))
        return -np.mean(yC * np.log(p) + (1 - yC) * np.log(1 - p))
    for lr in (0.01, 0.5, 4.0, 40.0):
        b0, b = logistic_gd(XC, yC, lr=lr, n_iter=300)
        print(f"lr={lr:<5} loss after 300 steps = {loss(b0, b):.5f}", b.round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m7")

---
## 8. PCA from scratch
*Medium · id `m8`*

<p><code>Z</code> is a (400, 6) array driven by two hidden factors. Write <code>pca(X, k)</code> that centres X (no scaling) and returns <code>(components, ratio)</code>: a (k, p) array whose rows are the top-k unit-length principal directions, and the share of total variance each explains.</p>
<p class=hint>SVD of the centred matrix: rows of Vᵀ are the directions, s²/Σs² the ratios. The sign of each component is arbitrary; the tests allow for it.</p>

**Theory to know cold**
<ul><li>Derive it: maximise wᵀΣw subject to wᵀw = 1. The Lagrangian gives Σw = λw, so w is the top eigenvector and the variance captured is λ.</li>
<li>Scale matters: PCA on raw data is dominated by large-variance columns. Standardise when units differ (that is PCA on the correlation matrix).</li>
<li>Choosing k: scree plot, cumulative explained variance, cross-validated reconstruction error.</li>
<li>In finance: the first PC of equity returns is roughly the market; of the yield curve, level, slope and curvature.</li>
<li>Others: factor analysis (models noise per variable), kernel PCA, ICA, t-SNE and UMAP (non-linear, for visualising; distances between clusters aren't meaningful), autoencoders, random projections.</li></ul>

In [ ]:
def pca(X, k):
    X = np.asarray(X, float)
    # your code here
    pass

# Sample run
try:
    c, r = pca(Z, 3)
    print(r.round(4))
    print(c.round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m8")

---
## 9. Explore returns with pandas and matplotlib
*Medium · id `m9`*

<p><code>rets</code> is a pandas DataFrame of daily returns for four tickers (dates as index), with a few missing values.</p>
<p>1. Write <code>summary(rets)</code> returning a DataFrame indexed by ticker with columns <code>ann_return</code> (mean × 252), <code>ann_vol</code> (std × √252), <code>sharpe</code> (their ratio) and <code>max_drawdown</code> (the most negative wealth / running peak − 1, with wealth = cumulative product of 1 + r, <b>treating missing returns as 0</b> for this column only). Sort by <code>sharpe</code>, highest first.</p>
<p>2. Write <code>plot(rets)</code> drawing cumulative wealth for each ticker and a correlation heatmap. Run shows your figures; plots aren't graded.</p>

**Theory to know cold**
<ul><li>The first five minutes of any data task: shape, dtypes, <code>describe()</code>, missing values, duplicates, the date range, outliers. Say out loud what you check and why.</li>
<li>pandas vs Polars: index-based vs expression-based, eager vs lazy, NaN vs null. Be able to do this problem in both.</li>
<li>matplotlib: <code>fig, ax = plt.subplots()</code>, label axes, add a title and legend, <code>imshow</code> or <code>matshow</code> for a heatmap with a colourbar.</li></ul>

In [ ]:
import matplotlib.pyplot as plt

def summary(rets: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

def plot(rets: pd.DataFrame):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    # your code here

# Sample run
try:
    print(rets.describe().round(5))
    print(summary(rets))
    plot(rets)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m9")

---
## 10. Fit and interpret a scikit-learn model
*Medium · id `m10`*

<p><code>credit</code> has <code>income, debt_ratio, age, late_payments</code> and a 0/1 <code>default</code> column.</p>
<p>Write <code>fit_default(df)</code>: split with <code>train_test_split(test_size=0.3, random_state=0, stratify=y)</code>, fit a <code>Pipeline</code> of <code>StandardScaler</code> then <code>LogisticRegression()</code> (defaults) on the training set, and return a dict with <code>"auc"</code> (ROC AUC on the test set) and <code>"odds_ratios"</code> (a dict of feature → exp(coefficient)).</p>
<p>Then explain each odds ratio in one sentence, as you would to an interviewer.</p>

**Theory to know cold**
<ul><li>With scaled inputs, an odds ratio is the multiplicative change in the odds of default per one standard deviation of that feature, holding the others fixed.</li>
<li><code>LogisticRegression</code> is ℓ2-regularised by default (C = 1), so coefficients are shrunk; scikit-learn gives no p-values. For inference use statsmodels <code>Logit</code>.</li>
<li>AUC = probability a random defaulter scores above a random non-defaulter. It ignores calibration; check a reliability curve or the Brier score too.</li>
<li>Fit the scaler inside the pipeline so the test set doesn't leak into it. Stratify when classes are imbalanced. Use cross-validation for model choice, the test set once.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

FEATURES = ["income", "debt_ratio", "age", "late_payments"]

def fit_default(df):
    # your code here
    pass

# Sample run
try:
    print(credit.describe().round(2))
    print(fit_default(credit))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m10")

---
## 11. Mean-variance with a Lagrangian
*Hard · id `m11`*

<p><code>mu</code> (5 expected returns) and <code>Sigma</code> (5 × 5 covariance) are given. Short selling is allowed. Write three functions, each returning a weight vector:</p>
<ul><li><code>min_variance(Sigma)</code>: minimise wᵀΣw subject to 1ᵀw = 1.</li>
<li><code>mv_target(mu, Sigma, target)</code>: minimise wᵀΣw subject to 1ᵀw = 1 and μᵀw = target.</li>
<li><code>tangency(mu, Sigma, rf)</code>: the maximum-Sharpe portfolio of risky assets.</li></ul>
<p class=hint>Use <code>np.linalg.solve(Sigma, ...)</code>. With A = 1ᵀΣ⁻¹1, B = 1ᵀΣ⁻¹μ, C = μᵀΣ⁻¹μ and D = AC − B², the target portfolio is w = [(C − Bt)Σ⁻¹1 + (At − B)Σ⁻¹μ] / D.</p>

**Theory to know cold**
<ul><li>Write the Lagrangian ½wᵀΣw − λ(1ᵀw − 1) − γ(μᵀw − t), take first-order conditions, solve for λ and γ. Be able to do it on a whiteboard.</li>
<li>The frontier variance is (At² − 2Bt + C)/D, a parabola in t; two-fund separation: every frontier portfolio mixes any two of them.</li>
<li>Tangency: w ∝ Σ⁻¹(μ − rf·1). With a risk-free asset every investor holds it in some proportion (CAPM's market portfolio).</li>
<li>In practice: estimation error in μ dominates, so weights are unstable. Fixes are shrinkage of Σ (Ledoit-Wolf), constraints, Black-Litterman, or minimum variance, which ignores μ.</li></ul>

In [ ]:
def min_variance(Sigma):
    pass

def mv_target(mu, Sigma, target):
    pass

def tangency(mu, Sigma, rf):
    pass

# Sample run
try:
    w0 = min_variance(Sigma); print("min var", w0.round(3), "vol", np.sqrt(w0 @ Sigma @ w0).round(4))
    w = mv_target(mu, Sigma, 0.09); print("target 9%", w.round(3), "vol", np.sqrt(w @ Sigma @ w).round(4))
    wt = tangency(mu, Sigma, 0.02); print("tangency", wt.round(3), "sharpe", ((wt @ mu - 0.02) / np.sqrt(wt @ Sigma @ wt)).round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m11")

---
## 12. Bayesian A/B test
*Easy · id `m12`*

<p>A had <code>xa</code> conversions out of <code>na</code>, B had <code>xb</code> out of <code>nb</code>. With independent Beta(a0, b0) priors, each conversion rate has a Beta posterior.</p>
<p>Write <code>prob_b_beats_a(xa, na, xb, nb, a0=1, b0=1, draws=200_000, seed=0)</code> returning P(p_B &gt; p_A | data) by Monte Carlo.</p>
<p class=hint>Posterior is Beta(a0 + x, b0 + n − x). Draw both with <code>rng.beta</code> and compare.</p>

**Theory to know cold**
<ul><li>Conjugacy: Beta prior × binomial likelihood = Beta posterior. Know the update and the posterior mean (a0 + x)/(a0 + b0 + n).</li>
<li>P(B &gt; A) is not a p-value: it answers the question people actually ask, but depends on the prior; with lots of data the prior washes out.</li>
<li>Decisions: expected loss of choosing B, credible intervals of the lift, and why "peeking" is less of a problem but not free in the Bayesian setup.</li>
<li>Other conjugate pairs: normal-normal (known variance), gamma-Poisson. MCMC when there's no conjugate form.</li></ul>

In [ ]:
def prob_b_beats_a(xa, na, xb, nb, a0=1, b0=1, draws=200_000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(prob_b_beats_a(200, 2000, 236, 2000))
    print(prob_b_beats_a(20, 200, 24, 200))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m12")

---
## 13. Bayesian linear regression = ridge
*Medium · id `m13`*

<p>Model y = Xβ + ε, ε ~ N(0, σ²I), prior β ~ N(0, τ²I), with σ² known and no intercept (assume X and y are centred).</p>
<p>Write <code>bayes_linreg(X, y, sigma2, tau2)</code> returning <code>(post_mean, post_cov)</code>. The tests also check that your posterior mean equals ridge with λ = σ²/τ².</p>
<p class=hint>Complete the square in the exponent: precision = XᵀX/σ² + I/τ², mean = precision⁻¹ Xᵀy / σ².</p>

**Theory to know cold**
<ul><li>Derive the posterior by completing the square; show the MAP equals the ridge estimate with λ = σ²/τ². A Laplace prior gives lasso as the MAP.</li>
<li>Posterior predictive for a new x: mean xᵀm, variance σ² + xᵀSx. The second term is parameter uncertainty, which grows away from the data.</li>
<li>As τ² → ∞ the prior goes flat and you get OLS back, with Cov = σ²(XᵀX)⁻¹.</li></ul>

In [ ]:
def bayes_linreg(X, y, sigma2, tau2):
    X, y = np.asarray(X, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    Xc, yc = X - X.mean(0), y - y.mean()
    m, S = bayes_linreg(Xc, yc, 1.0, 0.1)
    print(m.round(4)); print(np.sqrt(np.diag(S)).round(4))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m13")

---
## 14. Kalman filter: local level
*Medium · id `m14`*

<p>A noisy series <code>level_obs</code> follows a random-walk level: μₜ = μₜ₋₁ + ηₜ, ηₜ ~ N(0, q), observed as yₜ = μₜ + εₜ, εₜ ~ N(0, r). Some observations are <code>NaN</code>.</p>
<p>Write <code>kalman_level(y, q, r, m0=0.0, p0=1e6)</code> returning <code>(m, P)</code>: arrays of the filtered mean and variance of μₜ after seeing y₁…yₜ. When yₜ is missing, predict but skip the update.</p>
<p class=hint>Predict: P⁻ = P + q. Update: K = P⁻/(P⁻ + r), m = m + K(y − m), P = (1 − K)P⁻. Start from (m0, p0) before the first observation.</p>

**Theory to know cold**
<ul><li>The filter is Bayes' rule with Gaussians: predict (prior), update (posterior); K weighs the new data by relative uncertainty.</li>
<li>The signal-to-noise ratio q/r sets the smoothing: large q follows the data, small q averages; in steady state the local-level filter is an exponential moving average.</li>
<li>General form: xₜ = Fxₜ₋₁ + w, yₜ = Hxₜ + v, with matrix versions of the same five equations. Optimal (minimum MSE) for linear-Gaussian models; best linear filter otherwise.</li>
<li>q and r are usually estimated by maximising the likelihood from the prediction errors. Smoother (RTS) vs filter: the smoother uses future data too, so don't use it in a backtest.</li></ul>

In [ ]:
def kalman_level(y, q, r, m0=0.0, p0=1e6):
    y = np.asarray(y, float)
    m_out, P_out = np.empty(len(y)), np.empty(len(y))
    # your code here
    return m_out, P_out

# Sample run
try:
    m, P = kalman_level(level_obs, q=0.09, r=1.0)
    print(np.c_[level_obs[35:45], m[35:45], P[35:45]].round(3))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m14")

---
## 15. Kalman filter: a moving hedge ratio
*Hard · id `m15`*

<p>Two prices <code>hx</code> and <code>hy</code> are related by yₜ = βₜxₜ + εₜ, where the hedge ratio drifts: βₜ = βₜ₋₁ + ηₜ, ηₜ ~ N(0, q), εₜ ~ N(0, r).</p>
<p>Write <code>kalman_beta(x, y, q, r, b0=0.0, p0=10.0)</code> returning the array of filtered βₜ. Then compare it with a rolling 60-day OLS β.</p>
<p class=hint>Same filter, but the observation matrix is xₜ: K = P⁻xₜ / (xₜ²P⁻ + r), β = β + K(yₜ − xₜβ), P = (1 − Kxₜ)P⁻.</p>

**Theory to know cold**
<ul><li>This is the standard pairs-trading use: the spread yₜ − βₜxₜ, with βₜ from the filter, adapts without a lookback window.</li>
<li>Compared with rolling OLS: no window to choose, no cliff when an old point drops out, and an uncertainty band for β for free.</li>
<li>Adding an intercept makes the state two-dimensional (αₜ, βₜ) with H = [1, xₜ]. Be able to write F, H, Q and R for it.</li></ul>

In [ ]:
def kalman_beta(x, y, q, r, b0=0.0, p0=10.0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    # your code here
    pass

# Sample run
try:
    import matplotlib.pyplot as plt
    kb = kalman_beta(hx, hy, q=1e-4, r=2.25)
    roll = pd.Series(hy).rolling(60).cov(pd.Series(hx)) / pd.Series(hx).rolling(60).var()
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(kb, label="Kalman beta"); ax.plot(roll, label="rolling 60 OLS beta (with intercept)")
    ax.set_ylim(0.9, 1.5); ax.legend(); ax.set_title("Hedge ratio over time")
    print(kb[-5:].round(4))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m15")